In [1]:
# NOTEBOOK: 02_ocr.ipynb  -- cell 1 (check kernel first)
import sys
print(sys.executable)

/home/jovyan/receipt-env/bin/python


In [3]:
# NOTEBOOK: 02_ocr.ipynb  -- cell 1 (self-contained, no import needed)
import os, json
import numpy as np, easyocr
from datasets import load_dataset

def to_plain(obj):
    if isinstance(obj, np.integer):  return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.ndarray):  return obj.tolist()
    if isinstance(obj, (list, tuple)): return [to_plain(x) for x in obj]
    return obj

ds = load_dataset("naver-clova-ix/cord-v2")["train"]
reader = easyocr.Reader(["en"], gpu=True)
os.makedirs("data/interim/train", exist_ok=True)

for idx in range(3):
    raw = reader.readtext(np.array(ds[idx]["image"]))
    rec = {"idx": idx, "ground_truth": ds[idx]["ground_truth"],
           "ocr": [{"box": to_plain(b), "text": t, "conf": to_plain(c)} for b,t,c in raw]}
    with open(f"data/interim/train/{idx:04d}.json","w",encoding="utf-8") as f:
        json.dump(rec, f, ensure_ascii=False)
print("done, wrote 3 files")

done, wrote 3 files


In [4]:
# NOTEBOOK: 02_ocr.ipynb  -- cell 2
import json
sample = json.load(open("data/interim/train/0000.json"))
print("num OCR lines:", len(sample["ocr"]))
print("first line:", sample["ocr"][0])

num OCR lines: 86
first line: {'box': [[300, 366], [354, 366], [354, 392], [300, 392]], 'text': 'Nasi', 'conf': 0.9998831152915955}


In [5]:
# NOTEBOOK: 02_ocr.ipynb  -- cell 3 (full run, all 800, with checkpointing)
import os, json, time
import numpy as np, easyocr
from datasets import load_dataset

def to_plain(obj):
    if isinstance(obj, np.integer):  return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.ndarray):  return obj.tolist()
    if isinstance(obj, (list, tuple)): return [to_plain(x) for x in obj]
    return obj

ds = load_dataset("naver-clova-ix/cord-v2")["train"]
reader = easyocr.Reader(["en"], gpu=True)
os.makedirs("data/interim/train", exist_ok=True)

start = time.time()
done = skipped = 0
for idx in range(len(ds)):
    out_path = f"data/interim/train/{idx:04d}.json"
    if os.path.exists(out_path):          # checkpoint: skip finished ones
        skipped += 1
        continue
    raw = reader.readtext(np.array(ds[idx]["image"]))
    rec = {"idx": idx, "ground_truth": ds[idx]["ground_truth"],
           "ocr": [{"box": to_plain(b), "text": t, "conf": to_plain(c)} for b,t,c in raw]}
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(rec, f, ensure_ascii=False)
    done += 1
    if done % 50 == 0:                     # progress every 50 receipts
        rate = done / (time.time() - start)
        print(f"{done} done, {skipped} skipped, {rate:.1f}/sec")

print(f"FINISHED: {done} newly done, {skipped} already existed, {time.time()-start:.0f} sec total")

50 done, 3 skipped, 2.7/sec
100 done, 3 skipped, 2.5/sec
150 done, 3 skipped, 2.3/sec
200 done, 3 skipped, 2.2/sec
250 done, 3 skipped, 2.3/sec
300 done, 3 skipped, 2.3/sec
350 done, 3 skipped, 2.3/sec
400 done, 3 skipped, 2.3/sec
450 done, 3 skipped, 2.3/sec
500 done, 3 skipped, 2.3/sec
550 done, 3 skipped, 2.3/sec
600 done, 3 skipped, 2.3/sec
650 done, 3 skipped, 2.4/sec
700 done, 3 skipped, 2.3/sec
750 done, 3 skipped, 2.4/sec
FINISHED: 797 newly done, 3 already existed, 342 sec total


In [6]:
# NOTEBOOK: 02_ocr.ipynb  -- cell 4 (verify the output)
import os
files = os.listdir("data/interim/train")
print("files written:", len(files))

files written: 800
